# Ruby Laser Fluorescence

---

## Purpose

In this lab, you will:

**Chemistry Objectives:**
- Understand fluorescence lifetime and its temperature dependence
- Apply Boltzmann statistics to excited state populations
- Determine radiative and non-radiative rate constants
- Extract Arrhenius parameters (activation energy, frequency factor)

**Coding Objectives:**
- Write functions to automate repetitive data processing
- Apply linear regression to extract physical parameters
- Build and test physical models iteratively
- Work independently with minimal scaffolding (capstone lab)

## Estimated Time: 90-120 minutes

## Success Criteria
- [ ] Extracted fluorescence lifetime from a single decay curve
- [ ] Created `find_lifetime()` function that processes any run
- [ ] Generated τ vs T plot for all data files
- [ ] Calculated A_E from low-temperature data
- [ ] Calculated A_T from mid-temperature data
- [ ] Extracted activation energy and frequency factor from Arrhenius plot
- [ ] Built complete model and compared to experimental data

---

# Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

## Troubleshooting Guide

| Error | Likely Cause | Fix |
|-------|-------------|-----|
| `FileNotFoundError` | Wrong path to data files | Check that path is `'data/tek00XXch1.csv'` |
| `KeyError: 'TIME (s)'` | Wrong column names | Use `'TIME'` and `'CH1'` (no units in names) |
| `NameError: name 'data' is not defined` | Cells run out of order | Run cells from top to bottom |
| Negative lifetime | Wrong sign in slope calculation | τ = -1/slope (slope should be negative) |
| `curve_fit` doesn't converge | Bad initial guess | Adjust `guess_params` values |
| Empty arrays after filtering | Thresholds too strict | Adjust time/signal thresholds |

# Fluorescence Properties of Ruby

In this lab, we explore fluorescence of ruby (Al₂O₃ doped with Cr³⁺)—the first-ever lasing medium.

**Key Physics:**
- Cr³⁺ ions are excited by pump light, then emit fluorescent photons
- Fluorescence lifetime τ depends on temperature due to competing relaxation pathways
- By analyzing τ vs T, we can extract rate constants for radiative and non-radiative processes

**What we'll determine:**
- A_E: Radiative rate from ²E state
- A_T: Radiative rate from ⁴T₂ state  
- E_a: Activation energy for thermal relaxation
- A: Frequency factor for thermal relaxation

---

## PART 1 — Extracting Fluorescence Lifetime (25 pts)

For exponential decay I(t) = I₀ e^(-t/τ), taking the natural log gives:

ln(I) = ln(I₀) - t/τ

So plotting ln(I) vs t gives a line with slope = -1/τ.

### CODE TASK 1.1: Load and visualize one decay curve

In [ ]:
# ============================================
# SUBGOAL: Load and inspect data
# ============================================

data = pd.read_csv('data/tek0001CH1.csv', skiprows=20)
print(data.head())
print(f"Columns: {list(data.columns)}")

# Checkpoint: verify data loaded correctly
assert len(data) > 0, "Data file not loaded - check the path"
assert 'TIME' in data.columns, "Column 'TIME' not found - check skiprows value"
assert 'CH1' in data.columns, "Column 'CH1' not found - check CSV headers"

In [ ]:
# Plot the raw fluorescence decay
plt.figure(figsize=(10, 4))
plt.plot(data['TIME'], data['CH1'])
plt.xlabel('Time (s)')
plt.ylabel('Signal (V)')
plt.title('Raw Fluorescence Decay')
plt.grid(True)
plt.show()

### CODE TASK 1.2: Transform data for linear regression

We need to:
1. Shift signal so minimum is positive (can't take log of negative numbers)
2. Take natural logarithm
3. Identify the linear region

In [ ]:
# ============================================
# SUBGOAL: Transform data for linear fit
# ============================================

# Shift signal to be positive (add offset + small constant to avoid log(0))
data['normalized'] = data['CH1'] + abs(data['CH1'].min()) + 0.001

# Take natural logarithm
data['ln_signal'] = np.log(data['normalized'])

# Plot to identify linear region
plt.figure(figsize=(10, 4))
plt.plot(data['TIME'], data['ln_signal'])
plt.xlabel('Time (s)')
plt.ylabel('ln(Signal)')
plt.title('Log-transformed Signal - Identify Linear Region')
plt.grid(True)
plt.show()

### CODE TASK 1.3: Fit the linear region and extract lifetime

In [ ]:
# ============================================
# SUBGOAL: Filter to linear region and fit
# ============================================

# Set thresholds to select linear region (adjust based on your plot above)
time_thresh = 0.0  # Start time - adjust if needed
signal_thresh = 0.01  # Minimum signal - exclude noise floor

# Filter data
mask = (data['TIME'] > time_thresh) & (data['normalized'] > signal_thresh)
filtered = data[mask]

# Extract x and y for fitting
x = filtered['TIME']
y = filtered['ln_signal']

# Linear regression
slope, intercept = np.polyfit(x, y, 1)
lifetime_tau = -1 / slope

print(f"Slope: {slope:.2f}")
print(f"Lifetime τ = {lifetime_tau:.6f} s = {lifetime_tau*1000:.3f} ms")

# Checkpoint: verify lifetime is reasonable for ruby (~1-5 ms at room temp)
assert 0.0001 < lifetime_tau < 0.01, f"Lifetime {lifetime_tau:.4f}s seems unreasonable - check your fit"

In [ ]:
# ============================================
# SUBGOAL: Visualize fit quality
# ============================================

x_fit = np.linspace(x.min(), x.max(), 100)
y_fit = slope * x_fit + intercept

plt.figure(figsize=(10, 4))
plt.plot(x, y, 'b.', alpha=0.5, label='Data')
plt.plot(x_fit, y_fit, 'r-', linewidth=2, label=f'Fit: τ = {lifetime_tau*1000:.2f} ms')
plt.xlabel('Time (s)')
plt.ylabel('ln(Signal)')
plt.title('Linear Fit to Extract Lifetime')
plt.legend()
plt.grid(True)
plt.show()

### SHORT RESPONSE QUESTIONS

**Q1.1 (5 pts):** Why do we add the minimum value to our signal before taking the logarithm? Does this offset affect the extracted lifetime value? Explain.

**Q1.2 (5 pts):** The fluorescence lifetime changes with temperature, so the "linear region" will have different widths for different runs. How can you set thresholds that work for any arbitrary run without knowing the lifetime beforehand?

### ANSWERS

*Your answers here*

---

## PART 2 — Automating Data Processing (25 pts)

We have ~60 decay curves at different temperatures. Let's write a function to process them all.

### CODE TASK 2.1: Create the `find_lifetime` function

In [ ]:
def find_lifetime(run_number, plot=False):
    """
    Extract fluorescence lifetime from a data file.
    
    Args:
        run_number: Integer identifying which data file (0-61)
        plot: If True, display diagnostic plots
    
    Returns:
        lifetime_tau: Fluorescence lifetime in seconds
    """
    # ============================================
    # SUBGOAL: Load data file
    # ============================================
    filename = f'data/tek{run_number:04d}CH1.csv'  # :04d pads with zeros
    data = pd.read_csv(filename, skiprows=20)
    
    # ============================================
    # SUBGOAL: Transform data
    # ============================================
    # YOUR CODE HERE: Add offset and take log (copy from Part 1)
    data['normalized'] = None  # Replace with your code
    data['ln_signal'] = None   # Replace with your code
    
    # ============================================
    # SUBGOAL: Filter and fit
    # ============================================
    # YOUR CODE HERE: Set thresholds, filter data, fit line
    time_thresh = 0.0
    signal_thresh = 0.01
    
    # Filter, extract x and y, fit
    # ...
    
    slope, intercept = None, None  # Replace with np.polyfit result
    lifetime_tau = -1 / slope
    
    # ============================================
    # SUBGOAL: Optional plotting for QA
    # ============================================
    if plot:
        # YOUR CODE HERE: Plot data and fit line
        pass
    
    return lifetime_tau

### CODE TASK 2.2: Test your function on several runs

In [ ]:
# Test on first run (lowest temperature)
tau_1 = find_lifetime(1, plot=True)
print(f"Run 1: τ = {tau_1*1000:.3f} ms")

In [ ]:
# Test on a middle run
tau_30 = find_lifetime(30, plot=True)
print(f"Run 30: τ = {tau_30*1000:.3f} ms")

In [ ]:
# Test on last run (highest temperature)
tau_61 = find_lifetime(61, plot=True)
print(f"Run 61: τ = {tau_61*1000:.3f} ms")

### CODE TASK 2.3: Process all data and create τ vs T plot

In [ ]:
# ============================================
# SUBGOAL: Generate temperature and lifetime arrays
# ============================================

# Set these based on your experimental setup
max_index = 62  # Number of data files (0 to 61)
min_temperature = 300  # Starting temperature (K) - adjust as needed
temperature_step = 10  # Temperature increment between runs (K)

run_indices = list(range(1, max_index))
temperatures = np.array([min_temperature + (i-1) * temperature_step for i in run_indices])
lifetimes = np.array([find_lifetime(i) for i in run_indices])

# Checkpoint
assert len(temperatures) == len(lifetimes), "Arrays must have same length"
print(f"Processed {len(lifetimes)} data files")
print(f"Temperature range: {temperatures.min():.0f} K to {temperatures.max():.0f} K")
print(f"Lifetime range: {lifetimes.min()*1000:.3f} ms to {lifetimes.max()*1000:.3f} ms")

In [ ]:
# Plot τ vs T
plt.figure(figsize=(10, 6))
plt.scatter(temperatures, lifetimes * 1000, c='blue', alpha=0.7)
plt.xlabel('Temperature (K)')
plt.ylabel('Lifetime τ (ms)')
plt.title('Fluorescence Lifetime vs Temperature')
plt.grid(True)
plt.show()

### SHORT RESPONSE QUESTIONS

**Q2.1 (5 pts):** What are the benefits of automating the analysis with a function? What are the risks?

**Q2.2 (5 pts):** Did you encounter any issues when processing all files? How did you resolve them?

### ANSWERS

*Your answers here*

---

## PART 3 — Determining A_E and A_T (25 pts)

The lifetime depends on three rate constants:

$$\tau = \frac{1 + n_T/n_E}{A_E + [A_T + N(T)] \cdot n_T/n_E}$$

where $n_T/n_E = 8.311 \cdot e^{-3380/T}$ is the Boltzmann population ratio.

**Strategy:**
1. At low T: $n_T/n_E \approx 0$, so $\tau \approx 1/A_E$
2. At mid T: $N(T) \approx 0$, so we can solve for $A_T$

### CODE TASK 3.1: Calculate A_E from low-temperature data

In [ ]:
def calc_AE(tau):
    """Calculate A_E from lifetime (low-T approximation)."""
    return 1 / tau

def population_ratio(T):
    """Boltzmann population ratio n_T/n_E."""
    return 8.311 * np.exp(-3380 / T)

# Select low-temperature region where lifetime is roughly constant
low_T_mask = temperatures < 320  # Adjust based on your plot
AE = np.mean(calc_AE(lifetimes[low_T_mask]))

print(f"A_E = {AE:.1f} s⁻¹")
print(f"Corresponding lifetime: {1/AE*1000:.2f} ms")

In [ ]:
def lifetime_model_AE_only(T, AE):
    """Model with only A_E (constant lifetime)."""
    return np.ones_like(T) / AE

# Compare model to data
plt.figure(figsize=(10, 6))
plt.scatter(temperatures, lifetimes * 1000, label='Experimental', alpha=0.7)
plt.plot(temperatures, lifetime_model_AE_only(temperatures, AE) * 1000, 
         'r--', label=f'A_E only model (τ = {1/AE*1000:.2f} ms)')
plt.xlabel('Temperature (K)')
plt.ylabel('Lifetime (ms)')
plt.title('Comparison: A_E-only Model vs Data')
plt.legend()
plt.grid(True)
plt.show()

### CODE TASK 3.2: Calculate A_T from mid-temperature data

In [ ]:
def calc_AT(tau, T, AE):
    """Calculate A_T from lifetime and temperature (mid-T approximation, N(T)≈0)."""
    pop = population_ratio(T)
    # YOUR CODE HERE: Implement the formula
    # A_T = (1 + 1/pop) / tau - (1/pop) * AE
    return None  # Replace with your implementation

# Select mid-temperature region
mid_T_mask = (temperatures > 320) & (temperatures < 550)
AT_values = calc_AT(lifetimes[mid_T_mask], temperatures[mid_T_mask], AE)
AT = np.mean(AT_values)

print(f"A_T = {AT:.1f} s⁻¹")

In [ ]:
def lifetime_model_AE_AT(T, AE, AT):
    """Model with A_E and A_T (no thermal relaxation)."""
    pop = population_ratio(T)
    # YOUR CODE HERE: Implement τ = (1 + pop) / (AE + AT * pop)
    return None  # Replace with your implementation

# Compare model to data
plt.figure(figsize=(10, 6))
plt.scatter(temperatures, lifetimes * 1000, label='Experimental', alpha=0.7)
plt.plot(temperatures, lifetime_model_AE_AT(temperatures, AE, AT) * 1000, 
         'g-', linewidth=2, label='A_E + A_T model')
plt.xlabel('Temperature (K)')
plt.ylabel('Lifetime (ms)')
plt.title('Comparison: A_E + A_T Model vs Data')
plt.legend()
plt.grid(True)
plt.show()

### SHORT RESPONSE QUESTIONS

**Q3.1 (10 pts):** Compare the A_E-only and A_E+A_T models to the experimental data. At what temperature range does each model work well? Where does it fail? What does this tell you about the dominant relaxation pathways at different temperatures?

### ANSWERS

*Your answer here*

---

## PART 4 — Arrhenius Analysis of Thermal Relaxation (25 pts)

The non-radiative rate follows the Arrhenius equation:

$$N(T) = A \cdot e^{-E_a/kT}$$

Taking the log: $\ln(N) = \ln(A) - \frac{E_a}{k} \cdot \frac{1}{T}$

So plotting $\ln(N)$ vs $1/T$ gives slope $= -E_a/k$ and intercept $= \ln(A)$.

### CODE TASK 4.1: Extract N(T) from high-temperature data

In [ ]:
def calc_NT(tau, T, AE, AT):
    """Calculate N(T) from lifetime using full equation."""
    pop = population_ratio(T)
    # N(T) = (1 + 1/pop) / tau - (1/pop) * AE - AT
    # YOUR CODE HERE
    return None  # Replace with your implementation

# Select high-temperature region where N(T) is significant
high_T_mask = temperatures > 600  # Adjust as needed
NT_temperatures = temperatures[high_T_mask]
NT_values = calc_NT(lifetimes[high_T_mask], NT_temperatures, AE, AT)

# Checkpoint: N(T) should be positive
assert np.all(NT_values > 0), "N(T) values should be positive - check your calculation or threshold"

plt.figure(figsize=(10, 5))
plt.plot(NT_temperatures, NT_values, 'o-')
plt.xlabel('Temperature (K)')
plt.ylabel('N(T) (s⁻¹)')
plt.title('Non-radiative Rate vs Temperature')
plt.grid(True)
plt.show()

### CODE TASK 4.2: Arrhenius plot and linear fit

In [ ]:
# Transform for Arrhenius plot
log_NT = np.log(NT_values)
reciprocal_T = 1 / NT_temperatures

# Linear fit
slope, intercept = np.polyfit(reciprocal_T, log_NT, 1)

# Extract parameters
E_over_k = -slope  # E_a/k in Kelvin
freq_factor = np.exp(intercept)  # A in s⁻¹

# Convert to cm⁻¹
k_boltzmann = 0.695  # cm⁻¹/K
E_activation = E_over_k * k_boltzmann  # in cm⁻¹

print(f"Slope: {slope:.1f} K")
print(f"Intercept: {intercept:.2f}")
print(f"\nArrhenius parameters:")
print(f"  Activation energy E_a = {E_activation:.0f} cm⁻¹")
print(f"  Frequency factor A = {freq_factor:.2e} s⁻¹")

In [ ]:
# Plot Arrhenius fit
x_fit = np.linspace(reciprocal_T.min(), reciprocal_T.max(), 100)
y_fit = slope * x_fit + intercept

plt.figure(figsize=(10, 6))
plt.plot(reciprocal_T * 1000, log_NT, 'bo', label='Data')
plt.plot(x_fit * 1000, y_fit, 'r-', linewidth=2, 
         label=f'Fit: E_a = {E_activation:.0f} cm⁻¹')
plt.xlabel('1000/T (K⁻¹)')
plt.ylabel('ln(N(T))')
plt.title('Arrhenius Plot for Thermal Relaxation')
plt.legend()
plt.grid(True)
plt.show()

### CODE TASK 4.3: Build complete model and compare to data

In [ ]:
def NT_model(T, E_over_k, freq_factor):
    """Arrhenius model for N(T)."""
    return freq_factor * np.exp(-E_over_k / T)

def final_lifetime_model(T, AE, AT, E_over_k, freq_factor):
    """Complete model including all relaxation pathways."""
    pop = population_ratio(T)
    NT = NT_model(T, E_over_k, freq_factor)
    # τ = (1 + pop) / (AE + (AT + NT) * pop)
    # YOUR CODE HERE
    return None  # Replace with your implementation

# Calculate model predictions
model_lifetimes = final_lifetime_model(temperatures, AE, AT, E_over_k, freq_factor)

# Plot comparison
plt.figure(figsize=(12, 6))
plt.scatter(temperatures, lifetimes * 1000, label='Experimental', alpha=0.7, s=30)
plt.plot(temperatures, model_lifetimes * 1000, 'r-', linewidth=2, label='Complete Model')
plt.xlabel('Temperature (K)')
plt.ylabel('Lifetime (ms)')
plt.title('Final Model vs Experimental Data')
plt.legend()
plt.grid(True)
plt.show()

### SHORT RESPONSE QUESTIONS

**Q4.1 (5 pts):** How well does the complete model fit the experimental data? What might account for any remaining discrepancies?

**Q4.2 (5 pts):** Compare the complete physical model to the simple decaying exponential fit from Part 2. What are the advantages of each approach? When would you use each?

### ANSWERS

*Your answers here*

---

## Results Summary

Fill in your final values:

In [ ]:
print("=" * 50)
print("RESULTS SUMMARY")
print("=" * 50)
print(f"Radiative rate from ²E state:  A_E = {AE:.1f} s⁻¹")
print(f"Radiative rate from ⁴T₂ state: A_T = {AT:.1f} s⁻¹")
print(f"Activation energy:             E_a = {E_activation:.0f} cm⁻¹")
print(f"Frequency factor:              A   = {freq_factor:.2e} s⁻¹")
print("=" * 50)

---

# Reflection (10 pts)

### SHORT RESPONSE QUESTIONS

**Q5.1 (4 pts):** Summarize the physical meaning of each parameter you extracted (A_E, A_T, E_a, A). Is it surprising that simple fluorescence decay curves contain so much information?

**Q5.2 (3 pts):** We used approximations (assuming certain terms are negligible) to extract individual parameters. How did this "coarse-graining" approach enable us to learn more than we could from the full equation alone?

**Q5.3 (3 pts):** Reflect on how Python enabled this analysis. Could you have done this with 60+ files in Excel? What would have been different?

### ANSWERS

*Your answers here*

---

## References

1. Fonger, W. H., & Struck, C. W. (1970). Temperature dependences of Cr+3 radiative and nonradiative transitions in ruby. *Physical Review B*, 11(8), 3251.

2. [How Lasers Work - A Complete Guide](https://www.youtube.com/watch?v=_JOchLyNO_w) (YouTube)